# Notebook del Processing Job

**Maestría en Ciencia de Datos — Métodos de Gran Escala | ITAM**

Este notebook demuestra el flujo completo de Bring Your Own Container (BYOC) con scikit-learn

- Setup: sesión de SageMaker, IAM role, bucket y prefix.
- Carga del dataset a S3: sube tus datos crudos al bucket de SageMaker.
- Ejecución del Processing Job.
- Inspección del output: lee las primeras filas del CSV transformado desde S3 para verificar que el job fue exitoso.

> **Prerequisito:** Haber ejecutado `bash processing/build_and_push.sh sagemaker-processing-byoc` para que la imagen esté disponible en ECR.


## Setup {#setup}
Especifica el S3 bucket, prefixes y el IAM role para el processing job.

In [2]:
from time import gmtime, strftime
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
timestamp_prefix = strftime("%Y-%m-%d-%H-%M-%S", gmtime())

prefix = "sagemaker/processing-data"

if default_bucket_prefix:
    prefix = f"{default_bucket_prefix}/{prefix}"

input_prefix = prefix + "/input/raw"
input_preprocessed_prefix = prefix + "/input/preprocessed"

output_prefix = prefix + "/output"


input_container_path = "/opt/ml/processing/input"
output_container_path = "/opt/ml/processing/output"

print(f"Bucket: {bucket}")
print(f"Input S3 path: s3://{bucket}/{input_prefix}")
print(f"Output S3 path: s3://{bucket}/{output_prefix}")

Bucket: sagemaker-us-east-1-150215480648
Input S3 path: s3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw
Output S3 path: s3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/output


### Descarga del dataset y carga a Amazon Simple Storage Service (Amazon S3)

In [3]:
import boto3
import pandas as pd

# Ruta local de los datos
local_data_path = "../../data/raw"

s3 = boto3.client("s3")
region = sagemaker_session.boto_region_name
input_data = f"s3://{bucket}/{input_prefix}".format(region)
print(f"Usando ruta S3: {input_data}")


s3_data_uri = sagemaker_session.upload_data(
    path=local_data_path,
    bucket=bucket,
    key_prefix=input_prefix)
print(f" Datos cargados a: {s3_data_uri}\n")


# Verifiacación de datos subidos
response = sagemaker_session.boto_session.client("s3").list_objects_v2(
        Bucket=bucket, 
        Prefix=input_prefix
    )
print(f"\n Archivos en S3:")
if "Contents" in response:
    for obj in response["Contents"]:
        print(f"  ok {obj['Key']}")
        

Usando ruta S3: s3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw
 Datos cargados a: s3://sagemaker-us-east-1-150215480648/sagemaker/processing-data/input/raw


 Archivos en S3:
  ok sagemaker/processing-data/input/raw/item_categories.csv
  ok sagemaker/processing-data/input/raw/item_categories_en.csv
  ok sagemaker/processing-data/input/raw/items.csv
  ok sagemaker/processing-data/input/raw/items_en.csv
  ok sagemaker/processing-data/input/raw/sales_train.csv
  ok sagemaker/processing-data/input/raw/shops.csv
  ok sagemaker/processing-data/input/raw/shops_en.csv
  ok sagemaker/processing-data/input/raw/submission.csv
  ok sagemaker/processing-data/input/raw/test.csv


## Construcción del container {#container}

El container BYOC es una imagen Python slim con scikit-learn, pandas y numpy.
No requiere ningún proceso de bootstrapping — SageMaker inyecta y ejecuta el script
directamente con `python3`.

In [8]:
%cd ~/Arquitectura
!docker build --network sagemaker -t sagemaker-processing-byoc processing/container/

/home/sagemaker-user/Arquitectura
DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  29.18kB
Step 1/5 : FROM python:3.11-slim
 ---> 7c68b5683872
Step 2/5 : ENV PYTHONHASHSEED 0
 ---> Using cache
 ---> b0398c1d5ff1
Step 3/5 : ENV PYTHONIOENCODING UTF-8
 ---> Using cache
 ---> e4623bbf1b67
Step 4/5 : RUN pip install --no-cache-dir     numpy==2.4.2     pandas==3.0.0     pyarrow>=23.0.0     scikit-learn==1.8.0
 ---> Using cache
 ---> 265fcdf20739
Step 5/5 : LABEL com.amazon.studio.user.resources=true
 ---> Using cache
 ---> 5e1228a9d775
Successfully built 5e1228a9d775
Successfully tagged sagemaker-processing-byoc:latest


### Push a Amazon ECR

In [9]:
import boto3

account_id = boto3.client("sts").get_caller_identity().get("Account")
region = boto3.session.Session().region_name

ecr_repository = "sagemaker-sklearn-preprocess"
tag = ":latest"
uri_suffix = "amazonaws.com"

sklearn_repository_uri = "{}.dkr.ecr.{}.{}/{}".format(
    account_id, region, uri_suffix, ecr_repository + tag
)

In [11]:
# Create ECR repository and push docker image
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!aws ecr create-repository --repository-name $ecr_repository
!docker tag {ecr_repository + tag} $sklearn_repository_uri
!docker push $sklearn_repository_uri

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded

An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'sagemaker-sklearn-preprocess' already exists in the registry with id '150215480648'
The push refers to repository [150215480648.dkr.ecr.us-east-1.amazonaws.com/sagemaker-sklearn-preprocess]

71aa13c7: Preparing 
322086e6: Preparing 
c31ad48f: Preparing 
0bcd44fe: Preparing 
latest: digest: sha256:2db3ef07f16e025e0c1c5a63963130eacb7a5bddabab6d7c0e51052fe655cd73 size: 1372


## Script de preprocessing {#script}
El script `preprocess.py` ya fue desarrollado y se encuentra en `processing/container/preprocess.py`.

**¿Qué hace el script?**
- Carga y une los datos de ventas y tiendas.
- Limpieza de datos: elimina duplicados y descarta outliers.
- Consolidación y agrupación: genera una matriz con todas las combinaciones posibles de mes-tienda-producto.
- Ingenería de variables: integra los datos de prueba (mes 34) y genera las variables de historia meses anteriores (lags).
- División temporal de los datos: divide el dataset cronológicamente con el número de mes (entrenamiento <33, validación=33, inferencia=34).
- **Output**: guarda 3 archivos en formato .parquet (datos_entreno.parquet, datos_validacion.parquet y datos_inferencia.parquet) en la ruta /opt/ml/processing/output/ .

## Ejecutar el processing job {#run}

In [12]:
from sagemaker.processing import ScriptProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor = ScriptProcessor(
    base_job_name="sklearn-preprocessor",
    image_uri=sklearn_repository_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1200,
)

sklearn_processor.run(
    code="processing/container/preprocess.py",
    inputs=[
        ProcessingInput(
            source=s3_data_uri,
            destination=input_container_path,  
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="processed_data",
            source=output_container_path,  
            destination=f"s3://{bucket}/{output_prefix}"
        )
    ],
    logs=True,
)

print("Processing Job completado")

INFO:sagemaker:Creating processing-job with name sklearn-preprocessor-2026-03-16-20-57-27-536


........2026-03-16 20:58:44,519 - __main__ - INFO - ======================================================================
2026-03-16 20:58:44,519 - __main__ - INFO - Iniciando preprocesamiento de datos para SageMaker
2026-03-16 20:58:44,519 - __main__ - INFO - ======================================================================
2026-03-16 20:58:44,519 - __main__ - INFO - Input path:  /opt/ml/processing/input
2026-03-16 20:58:44,519 - __main__ - INFO - Output path: /opt/ml/processing/output
2026-03-16 20:58:44,519 - __main__ - INFO - 
[1/6] Cargando datasets raw...
2026-03-16 20:58:44,520 - __main__ - INFO -   ✓ items: /opt/ml/processing/input/items_en.csv
2026-03-16 20:58:44,520 - __main__ - INFO -   ✓ categories: /opt/ml/processing/input/item_categories_en.csv
2026-03-16 20:58:44,520 - __main__ - INFO -   ✓ shops: /opt/ml/processing/input/shops_en.csv
2026-03-16 20:58:44,520 - __main__ - INFO -   ✓ train: /opt/ml/processing/input/sales_train.csv
2026-03-16 20:58:44,520 - __main__ -

## Inspeccionar el output {#inspect}

Revisa las primeras filas del dataset transformado para verificar que el preprocessing fue exitoso.

In [13]:
import pandas as pd

s3_train_features = f"s3://{bucket}/{output_prefix}/datos_entreno.parquet"

df_train = pd.read_parquet(s3_train_features)
print("Shape set de entrenamiento:", df_train.shape)
display(df_train.head())



Shape set de entrenamiento: (10675632, 9)


,date_block_num,shop_id,item_id,item_cnt_month,item_cnt_month_mes_ant_1,item_cnt_month_mes_ant_2,item_cnt_month_mes_ant_3,item_cnt_month_mes_ant_12,item_category_id
0,0,0,19,0.0,0.0,0.0,0.0,0.0,40
1,0,0,27,0.0,0.0,0.0,0.0,0.0,19
2,0,0,28,0.0,0.0,0.0,0.0,0.0,30
3,0,0,29,0.0,0.0,0.0,0.0,0.0,23
4,0,0,32,6.0,0.0,0.0,0.0,0.0,40


In [14]:
s3_val_features = f"s3://{bucket}/{output_prefix}/datos_validacion.parquet"

df_train = pd.read_parquet(s3_val_features)
print("Shape set de validacion:", df_train.shape)
display(df_train.head())

Shape set de validacion: (238172, 9)


,date_block_num,shop_id,item_id,item_cnt_month,item_cnt_month_mes_ant_1,item_cnt_month_mes_ant_2,item_cnt_month_mes_ant_3,item_cnt_month_mes_ant_12,item_category_id
0,33,2,30,0.0,0.0,0.0,0.0,0.0,40
1,33,2,31,1.0,0.0,0.0,0.0,0.0,37
2,33,2,32,0.0,0.0,1.0,0.0,2.0,40
3,33,2,33,0.0,1.0,0.0,1.0,0.0,37
4,33,2,40,0.0,0.0,0.0,0.0,0.0,57


In [15]:
s3_infer_features = f"s3://{bucket}/{output_prefix}/datos_inferencia.parquet"

df_train = pd.read_parquet(s3_infer_features)
print("Shape set de inferencia:", df_train.shape)
display(df_train.head())


Shape set de inferencia: (214200, 9)


,date_block_num,shop_id,item_id,item_cnt_month,item_cnt_month_mes_ant_1,item_cnt_month_mes_ant_2,item_cnt_month_mes_ant_3,item_cnt_month_mes_ant_12,item_category_id
0,34,5,5037,0.0,0.0,1.0,3.0,1.0,19
1,34,5,5320,0.0,0.0,0.0,0.0,0.0,55
2,34,5,5233,0.0,1.0,3.0,1.0,0.0,19
3,34,5,5232,0.0,0.0,0.0,1.0,0.0,23
4,34,5,5268,0.0,0.0,0.0,0.0,0.0,20
